In [ ]:
import numpy as np
import pandas as pd 

import os
import kagglehub

In [4]:
!pip uninstall -y mamba-ssm causal-conv1d -q
!TORCH_CUDA_ARCH_LIST="7.5" pip install causal-conv1d --no-build-isolation --no-cache-dir -q
!TORCH_CUDA_ARCH_LIST="7.5" pip install mamba-ssm --no-build-isolation --no-cache-dir -q
!pip install datasets transformers -q

  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 2.9 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (pyproject.toml) ... done


In [5]:
import torch
from mamba_ssm import Mamba
m = Mamba(d_model=64, d_state=16, expand=2).cuda()
out = m(torch.randn(2, 10, 64).cuda())
print("Output shape:", out.shape)
print("Number of GPUs:", torch.cuda.device_count())

print(f"GPUs available: {torch.cuda.device_count()}")

import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF authenticated.")
except Exception as e:
    print(f"No HF token set up ({e}) — proceeding unauthenticated, just rate-limited.")

Mamba fused kernel works: torch.Size([2, 10, 64])
GPUs available: 2
No HF token set up (Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 132393154 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.) — proceeding unauthenticated, just rate-limited.


In [6]:
import os
import math
import time
from dataclasses import dataclass, asdict
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# Note : 
# Try to use the optimized CUDA implementation of Mamba.
# If it is not available, use the PyTorch selective scan below.
try:
    from mamba_ssm import Mamba as _CudaMamba
    USE_MAMBA_CUDA = True
except ImportError:
    USE_MAMBA_CUDA = False


class SelectiveScan(nn.Module):
    def __init__(self, d_inner: int, d_state: int = 16):
        super().__init__()
        self.d_inner, self.d_state = d_inner, d_state
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.x_proj = nn.Linear(d_inner, d_state * 2 + d_inner, bias=False)
        self.dt_proj = nn.Linear(d_inner, d_inner, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B_, L, D_ = x.shape
        A = -torch.exp(self.A_log.float())
        proj = self.x_proj(x)
        Bmat, Cmat, dt_in = torch.split(proj, [self.d_state, self.d_state, self.d_inner], dim=-1)
        delta = F.softplus(self.dt_proj(dt_in))
        h = torch.zeros(B_, D_, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dA = torch.exp(delta[:, t].unsqueeze(-1) * A)
            dBx = delta[:, t].unsqueeze(-1) * Bmat[:, t].unsqueeze(1) * x[:, t].unsqueeze(-1)
            h = dA * h + dBx
            ys.append((h * Cmat[:, t].unsqueeze(1)).sum(-1) + self.D * x[:, t])
        return torch.stack(ys, dim=1)


class MambaBlock(nn.Module):
    """Causal Mamba block, pre-norm + residual."""
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2, use_grad_checkpoint: bool = True):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.use_grad_checkpoint = use_grad_checkpoint
        d_inner = expand * d_model
        if USE_MAMBA_CUDA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.in_proj = nn.Linear(d_model, d_inner)
            self.out_proj = nn.Linear(d_inner, d_model)
            self.scan = SelectiveScan(d_inner, d_state)

    def _forward_impl(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if USE_MAMBA_CUDA:
            out = self.mamba(h)
        else:
            out = self.out_proj(self.scan(self.in_proj(h)))
        return x + out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(self._forward_impl, x, use_reentrant=False)
        return self._forward_impl(x)


@dataclass
class MambaConfig:
    vocab_size: int = 50257
    block_size: int = 1024
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2
    use_grad_checkpoint: bool = True


class MambaLM(nn.Module):
    def __init__(self, cfg: MambaConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, d_state=cfg.d_state, expand=cfg.expand,
                       use_grad_checkpoint=cfg.use_grad_checkpoint)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight  # weight tying

        nn.init.normal_(self.embed.weight, mean=0.0, std=0.02)

        if not USE_MAMBA_CUDA:
            self.apply(self._init_weights)
            for name, p in self.named_parameters():
                if name.endswith("out_proj.weight"):
                    nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss


def count_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())


def build_packed_stream(dataset_name, dataset_config, split, tokenizer, block_size, shuffle_buffer=10_000, seed=42):
    from datasets import load_dataset
    ds = load_dataset(dataset_name, dataset_config, split=split, streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=shuffle_buffer)
    eos_id = tokenizer.eos_token_id

    def token_stream():
        buffer = []
        for example in ds:
            text = example.get("text", "")
            if not text:
                continue
            ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            buffer.extend(ids + [eos_id])
            while len(buffer) >= block_size + 1:
                chunk = buffer[: block_size + 1]
                buffer = buffer[block_size:]
                yield {
                    "input_ids": torch.tensor(chunk[:-1], dtype=torch.long),
                    "labels": torch.tensor(chunk[1:], dtype=torch.long),
                }

    class PackedIterableDataset(torch.utils.data.IterableDataset):
        def __iter__(self):
            return token_stream()

    return PackedIterableDataset()


def build_dataloader(dataset_name, dataset_config, split, tokenizer, block_size, batch_size, shuffle_buffer=10_000):
    packed = build_packed_stream(dataset_name, dataset_config, split, tokenizer, block_size, shuffle_buffer)

    def collate(batch):
        return {
            "input_ids": torch.stack([b["input_ids"] for b in batch]),
            "labels": torch.stack([b["labels"] for b in batch]),
        }
    return torch.utils.data.DataLoader(packed, batch_size=batch_size, collate_fn=collate, num_workers=2)


def save_checkpoint(path, model, optimizer, scheduler, step, cfg, best_loss):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp_path = path + ".tmp"
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    torch.save({
        "step": step,
        "model_state": raw_model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "cfg": asdict(cfg),
        "best_loss": best_loss,
    }, tmp_path)
    os.replace(tmp_path, path)


def load_checkpoint(path, model, optimizer, scheduler, device):
    ckpt = torch.load(path, map_location=device)
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    return ckpt["step"], ckpt["best_loss"]


def setup_hf_auth() -> bool:
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token:
        os.environ["HF_TOKEN"] = token
        try:
            from huggingface_hub import login
            login(token=token, add_to_git_credential=False)
            print("Authenticated with Hugging Face Hub.")
        except Exception as e:
            print(f"[warning] found a token but login() failed: {e}")
        return True
    else:
        print("[warning] No HF_TOKEN found — proceeding unauthenticated.")
        return False


def find_latest_checkpoint(checkpoint_dir: str) -> Optional[str]:
    import glob
    candidates = []
    local = os.path.join(checkpoint_dir, "latest.pt")
    if os.path.exists(local):
        candidates.append(local)
    candidates.extend(glob.glob("/kaggle/input/**/latest.pt", recursive=True))
    if not candidates:
        return None
    candidates.sort(key=os.path.getmtime, reverse=True)
    if len(candidates) > 1:
        print(f"Found {len(candidates)} checkpoints, using most recent: {candidates[0]}")
    return candidates[0]


MAX_STEPS = 50_000

DEFAULT_ARGS = dict(
    dataset="wikitext",
    config="wikitext-103-raw-v1",
    block_size=512,
    d_model=384,
    n_layer=16,
    batch_size=32,
    lr=3e-4,
    warmup_steps=500,
    steps=40_000,
    shuffle_buffer=10_000,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=500,
    log_every=25,
    resume_from=None,
    use_grad_checkpoint=True,
)


def quick_param_count(d_model, n_layer, vocab_size=50257, expand=2, d_state=16):
    cfg = MambaConfig(vocab_size=vocab_size, d_model=d_model, n_layer=n_layer, expand=expand, d_state=d_state)
    m = MambaLM(cfg)
    n_params = count_params(m)
    print(f"d_model={d_model:4d}  n_layer={n_layer:2d}  ->  {n_params/1e6:.1f}M params")
    del m
    return n_params


def make_args(**overrides):
    from types import SimpleNamespace
    cfg = {**DEFAULT_ARGS, **overrides}
    return SimpleNamespace(**cfg)


def train(args):
    if args.steps > MAX_STEPS:
        print(f"[warning] steps={args.steps} exceeds MAX_STEPS={MAX_STEPS}; clamping.")
        args.steps = MAX_STEPS

    setup_hf_auth()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cfg = MambaConfig(vocab_size=tokenizer.vocab_size, block_size=args.block_size,
                         d_model=args.d_model, n_layer=args.n_layer,
                         use_grad_checkpoint=getattr(args, "use_grad_checkpoint", True))
    model = MambaLM(cfg).to(device)
    print(f"Model params: {count_params(model)/1e6:.1f}M  (device={device}, fused_kernel={HAS_CUDA_MAMBA})")
    if not USE_MAMBA_CUDA:
        print("[warning] fused mamba_ssm kernel not active — training will be very slow.")

    n_gpus = torch.cuda.device_count()
    if device == "cuda" and n_gpus > 1:
        print(f"Using {n_gpus} GPUs via DataParallel — batch_size={args.batch_size} splits to "
              f"~{args.batch_size // n_gpus} per GPU.")
        model = nn.DataParallel(model)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.1, betas=(0.9, 0.95))
    warmup_steps = min(args.warmup_steps, max(1, args.steps - 1))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=args.lr, total_steps=args.steps, pct_start=warmup_steps / args.steps)

    start_step, best_loss = 0, float("inf")
    latest_ckpt = os.path.join(args.checkpoint_dir, "latest.pt")
    resume_path = args.resume_from or find_latest_checkpoint(args.checkpoint_dir)
    if resume_path:
        start_step, best_loss = load_checkpoint(resume_path, model, opt, sched, device)
        print(f"Resumed from {resume_path} at step {start_step} (best_loss={best_loss:.4f})")
    else:
        print("No existing checkpoint found — starting fresh from step 0.")

    loader = build_dataloader(args.dataset, args.config, "train", tokenizer,
                               cfg.block_size, args.batch_size, args.shuffle_buffer)
    data_iter = iter(loader)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    model.train()
    t0 = time.time()
    for step in range(start_step, args.steps):
        step_t0 = time.time()
        batch = next(data_iter)
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        opt.zero_grad()
        with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=(device == "cuda")):
            logits, _ = model(input_ids)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()

        if step < start_step + 5 or step % args.log_every == 0:
            ppl = math.exp(min(loss.item(), 20))
            elapsed = time.time() - t0
            step_time = time.time() - step_t0
            print(f"step {step:6d}/{args.steps}  loss {loss.item():.4f}  ppl {ppl:.1f}  "
                  f"lr {sched.get_last_lr()[0]:.2e}  step_time {step_time:.2f}s  total {elapsed:.0f}s")

        if step % args.checkpoint_every == 0 and step > start_step:
            save_checkpoint(latest_ckpt, model, opt, sched, step, cfg, min(best_loss, loss.item()))
            if loss.item() < best_loss:
                best_loss = loss.item()
                save_checkpoint(os.path.join(args.checkpoint_dir, "best.pt"), model, opt, sched, step, cfg, best_loss)
            print(f"  [checkpoint saved at step {step}]")

    save_checkpoint(latest_ckpt, model, opt, sched, args.steps, cfg, best_loss)
    print("Training complete, final checkpoint saved.")
    return model

print("Cell 4 loaded OK.")

Cell 4 loaded OK.


In [7]:
quick_param_count(d_model=384, n_layer=16)

d_model= 384  n_layer=16  ->  34.7M params


34733184

In [ ]:
args = make_args(dataset="wikitext", config="wikitext-103-raw-v1",
                  d_model=384, n_layer=16, steps=40_000, checkpoint_every=500)
model = train(args)

[warning] No HF_TOKEN found — proceeding unauthenticated.
Model params: 34.7M  (device=cuda, fused_kernel=True)
Using 2 GPUs via DataParallel — batch_size=32 splits to ~16 per GPU.
Resumed from /kaggle/working/checkpoints/latest.pt at step 7000 (best_loss=3.8977)


README.md: 0.00B [00:00, ?B/s]

Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.


step   7000/40000  loss 3.9462  ppl 51.7  lr 2.80e-04  step_time 6.66s  total 7s
step   7001/40000  loss 4.0183  ppl 55.6  lr 2.80e-04  step_time 1.11s  total 8s
step   7002/40000  loss 4.0356  ppl 56.6  lr 2.80e-04  step_time 0.62s  total 8s
step   7003/40000  loss 4.0601  ppl 58.0  lr 2.80e-04  step_time 0.61s  total 9s
step   7004/40000  loss 4.0556  ppl 57.7  lr 2.80e-04  step_time 0.61s  total 10s
step   7025/40000  loss 3.9281  ppl 50.8  lr 2.80e-04  step_time 0.62s  total 23s
step   7050/40000  loss 4.0280  ppl 56.1  lr 2.80e-04  step_time 0.64s  total 38s
step   7075/40000  loss 4.0970  ppl 60.2  lr 2.80e-04  step_time 0.63s  total 54s
step   7100/40000  loss 4.0392  ppl 56.8  lr 2.80e-04  step_time 0.63s  total 70s
step   7125/40000  loss 3.8160  ppl 45.4  lr 2.80e-04  step_time 0.61s  total 85s
step   7150/40000  loss 3.7103  ppl 40.9  lr 2.79e-04  step_time 0.62s  total 100s
step   7175/40000  loss 4.0275  ppl 56.1  lr 2.79e-04  step_time 0.63s  total 116s
step   7200/40000 

Token indices sequence length is longer than the specified maximum sequence length for this model (1132 > 1024). Running this sequence through the model will result in indexing errors


step   7450/40000  loss 3.9002  ppl 49.4  lr 2.78e-04  step_time 0.63s  total 286s
step   7475/40000  loss 3.8850  ppl 48.7  lr 2.77e-04  step_time 0.62s  total 302s
step   7500/40000  loss 3.8077  ppl 45.0  lr 2.77e-04  step_time 0.61s  total 317s
  [checkpoint saved at step 7500]
step   7525/40000  loss 3.9124  ppl 50.0  lr 2.77e-04  step_time 0.63s  total 335s
step   7550/40000  loss 3.9756  ppl 53.3  lr 2.77e-04  step_time 0.62s  total 350s
step   7575/40000  loss 3.9080  ppl 49.8  lr 2.77e-04  step_time 0.61s  total 366s
step   7600/40000  loss 3.9269  ppl 50.7  lr 2.77e-04  step_time 0.62s  total 382s
step   7625/40000  loss 3.8954  ppl 49.2  lr 2.77e-04  step_time 0.62s  total 397s
step   7650/40000  loss 3.7877  ppl 44.2  lr 2.76e-04  step_time 0.62s  total 412s
step   7675/40000  loss 3.7811  ppl 43.9  lr 2.76e-04  step_time 0.62s  total 428s
step   7700/40000  loss 3.8240  ppl 45.8  lr 2.76e-04  step_time 0.62s  total 443s
step   7725/40000  loss 3.9470  ppl 51.8  lr 2.76e-04